# fsaverage6 collision-group profile QC

This notebook summarizes existing collision-group metrics. Each row is one subject × hemisphere × fsaverage6 target vertex group; profile metrics use complete 21-point MD profiles.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/results/native_surface_analysis/collision_profile_analysis_pilot/shards')
files=sorted(ROOT.glob('D*/tables/collision_group_metrics.csv'))
metrics=pd.concat([pd.read_csv(p) for p in files],ignore_index=True)
print(f'subjects={metrics.subject_id.nunique():,}; groups={len(metrics):,}; files={len(files):,}')
metrics['multiplicity_label']=metrics.n_columns_mapped.map(lambda x:str(int(x)) if x<6 else '>=6')
display(metrics.head())

## 1. Multiplicity and denominator

Multiplicity percentages use covered fsaverage6 target vertices as the denominator. Uncovered vertices are not included here.

In [ ]:
group_counts=metrics.groupby('multiplicity_label',sort=False).agg(covered_vertices=('target_vertex_id','size'),groups_with_2plus_valid_profiles=('n_profiles_valid',lambda x:int((x>=2).sum())),mixed_label_groups=('mixed_label','sum')).reset_index()
labels=['1','2','3','4','5','>=6']
group_counts=group_counts.set_index('multiplicity_label').reindex(labels).reset_index()
group_counts['covered_vertex_pct']=100*group_counts['covered_vertices']/group_counts['covered_vertices'].sum()
group_counts['profile_eligible_pct']=100*group_counts['groups_with_2plus_valid_profiles']/group_counts['covered_vertices']
group_counts['mixed_label_pct']=100*group_counts['mixed_label_groups']/group_counts['covered_vertices']
display(group_counts.round(2))
fig,ax=plt.subplots(figsize=(5.5,3.2))
sns.barplot(data=group_counts,x='multiplicity_label',y='covered_vertex_pct',color='#0072B2',order=labels,ax=ax)
ax.set_xlabel('Native columns mapped to one fsaverage6 vertex')
ax.set_ylabel('All covered vertices (%)')
ax.set_title('Multiplicity among all covered vertices')
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
conflict=group_counts[group_counts['multiplicity_label'].isin(['2','3','4','5','>=6'])].copy()
conflict_labels=['2','3','4','5','>=6']
conflict['conflict_vertex_pct']=100*conflict['covered_vertices']/conflict['covered_vertices'].sum()
conflict=conflict.set_index('multiplicity_label').reindex(conflict_labels).reset_index()
display(conflict[['multiplicity_label','covered_vertices','covered_vertex_pct','conflict_vertex_pct','mixed_label_pct']].round(2))
fig,ax=plt.subplots(figsize=(5.5,3.2))
sns.barplot(data=conflict,x='multiplicity_label',y='conflict_vertex_pct',color='#D55E00',order=conflict_labels,ax=ax)
ax.set_xlabel('Native columns mapped to one fsaverage6 vertex')
ax.set_ylabel('Conflict vertices (%)')
ax.set_title('Multiplicity distribution within conflict vertices')
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

## 2. Profile similarity and native spatial span

In [ ]:
eligible=metrics[metrics.n_profiles_valid>=2].copy()
plot_cols=['profile_pearson_median','profile_mae_median','profile_rmse_median','native_anchor_distance_median']
metric_labels={'profile_pearson_median':'Pearson correlation','profile_mae_median':'MAE','profile_rmse_median':'RMSE','native_anchor_distance_median':'Native anchor distance (mm)'}
long=eligible.melt(id_vars=['multiplicity_label'],value_vars=plot_cols,var_name='metric',value_name='value').dropna()
order=['2','3','4','5','>=6']
fig,axes=plt.subplots(2,2,figsize=(8,6)); axes=axes.ravel()
for ax,(name,g) in zip(axes, long.groupby('metric')):
    low,high=np.percentile(g.value,[1,99])
    sns.boxplot(data=g,x='multiplicity_label',y='value',order=order,showfliers=False,color='#D9EAF2',ax=ax)
    if name == 'profile_pearson_median':
        ax.set_ylim(-1, 1)
    else:
        ax.set_ylim(low, high)
    ax.set_title(metric_labels[name])
    ax.set_xlabel('Native columns per fsaverage6 vertex')
    ax.set_ylabel('')
    if name in ('profile_mae_median','profile_rmse_median'):
        ax.ticklabel_format(axis='y',style='sci',scilimits=(0,0))
    ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

## 3. Tail distributions

In [ ]:
tail_specs={'profile_pearson_median':('Pearson correlation',lambda x:np.mean(x<=x[:,None],axis=0) if False else None),'profile_mae_median':('MAE',None),'profile_rmse_median':('RMSE',None),'native_anchor_distance_median':('Native anchor distance median (mm)',None)}
fig,axes=plt.subplots(2,2,figsize=(8,6)); axes=axes.ravel()
for ax,(metric,(label,_)) in zip(axes,tail_specs.items()):
    for multiplicity in order:
        values=np.sort(eligible.loc[eligible.multiplicity_label.eq(multiplicity),metric].dropna().to_numpy())
        if not len(values): continue
        if metric=='profile_pearson_median':
            y=np.arange(1,len(values)+1)/len(values)
            ax.plot(values,y,label=multiplicity,linewidth=1.2)
            ax.set_ylabel('P(r ≤ x)')
        else:
            x=values
            y=1-np.arange(1,len(values)+1)/len(values)
            ax.step(x,y,where='post',label=multiplicity,linewidth=1.2)
            ax.set_ylabel('P(value > x)')
    ax.set_xlabel(label)
    ax.set_title(label)
    ax.legend(title='Multiplicity\n(n effective)',frameon=False,fontsize=6)
    if metric in ('profile_mae_median','profile_rmse_median'):
        ax.ticklabel_format(axis='x',style='sci',scilimits=(0,0))
    ax.spines[['top','right']].set_visible(False)
fig.suptitle('Tail distributions over the complete valid range; denominator is effective n per multiplicity',fontsize=9,y=1.02)
fig.tight_layout()
plt.show()

## 4. Subject-level summary

In [ ]:
subject_summary=metrics.groupby('subject_id').agg(collision_groups=('target_vertex_id','size'),groups_with_2plus_profiles=('n_profiles_valid',lambda x:int((x>=2).sum())),mixed_label_groups=('mixed_label','sum'),median_profile_r=('profile_pearson_median','median'),median_mae=('profile_mae_median','median'),median_rmse=('profile_rmse_median','median'),median_anchor_span=('native_anchor_distance_median','median')).reset_index()
subject_summary['eligible_group_pct']=100*subject_summary.groups_with_2plus_profiles/subject_summary.collision_groups
subject_summary['mixed_label_pct']=100*subject_summary.mixed_label_groups/subject_summary.collision_groups
display(subject_summary.round(4))

## 5. Label mixing

In [ ]:
conflict_metrics=metrics[metrics.n_columns_mapped>=2].copy()
conflict_order=['2','3','4','5','>=6']
plot=conflict_metrics.groupby(['multiplicity_label','mixed_label']).size().reset_index(name='groups')
plot['pct']=100*plot['groups']/plot.groupby('multiplicity_label')['groups'].transform('sum')
fig,ax=plt.subplots(figsize=(5.5,3.5))
sns.barplot(data=plot,x='multiplicity_label',y='pct',hue='mixed_label',order=conflict_order,palette={False:'#0072B2',True:'#D55E00'},ax=ax)
ax.set_xlabel('Multiplicity')
ax.set_ylabel('Groups (%)')
ax.set_title('Mixed-label groups within conflict vertices')
ax.legend(title='Mixed label',frameon=False,bbox_to_anchor=(1.02,1),loc='upper left')
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

## 6. Representative and tail groups

In [ ]:
eligible=metrics[metrics.n_profiles_valid>=2].copy()
small=eligible.nsmallest(10,'profile_mae_median')[['subject_id','hemisphere','source_region','target_vertex_id','n_columns_mapped','n_profiles_valid','profile_pearson_median','profile_mae_median','profile_rmse_median','native_anchor_distance_max','mixed_label']]
large=eligible.nlargest(10,'profile_mae_median')[['subject_id','hemisphere','source_region','target_vertex_id','n_columns_mapped','n_profiles_valid','profile_pearson_median','profile_mae_median','profile_rmse_median','native_anchor_distance_max','mixed_label']]
print('Smallest profile differences'); display(small.round(6)); print('Largest profile differences'); display(large.round(6))

In [ ]:
id_cols=['subject_id','hemisphere','target_vertex_id','multiplicity_label','mixed_label','n_profiles_valid']
pearson_tail=eligible.nsmallest(20,'profile_pearson_median')[id_cols+['profile_pearson_median','profile_mae_median','profile_rmse_median','native_anchor_distance_median','native_anchor_distance_max']].assign(tail_type='lowest Pearson')
rmse_tail=eligible.nlargest(20,'profile_rmse_median')[id_cols+['profile_pearson_median','profile_mae_median','profile_rmse_median','native_anchor_distance_median','native_anchor_distance_max']].assign(tail_type='highest RMSE')
distance_tail=eligible.nlargest(20,'native_anchor_distance_max')[id_cols+['profile_pearson_median','profile_mae_median','profile_rmse_median','native_anchor_distance_median','native_anchor_distance_max']].assign(tail_type='maximum native anchor distance')
tail_groups=pd.concat([pearson_tail,rmse_tail,distance_tail],ignore_index=True)
tail_output=ROOT.parent/'tail_groups_to_review.csv'
tail_groups.to_csv(tail_output,index=False)
print('待检查尾部组：不自动排除，仅用于回溯检查')
print(f'Saved: {tail_output}')
display(tail_groups.round(6))

## 7. Linking profile differences, spatial span, label mixing, and boundaries

In [ ]:
eligible=metrics[metrics.n_profiles_valid>=2].copy()
relationship_cols=['profile_pearson_median','profile_mae_median','profile_rmse_median','native_anchor_distance_median','native_anchor_distance_max']
relationship_corr=eligible[relationship_cols].corr(method='spearman')
display(relationship_corr.round(3))
fig,axes=plt.subplots(1,2,figsize=(8,3.2))
sns.scatterplot(data=eligible,x='native_anchor_distance_max',y='profile_rmse_median',hue='mixed_label',palette={False:'#0072B2',True:'#D55E00'},alpha=.35,s=12,ax=axes[0])
axes[0].set_xlabel('Maximum native anchor distance (mm)')
axes[0].set_ylabel('Profile RMSE')
axes[0].set_title('RMSE versus maximum spatial span')
sns.boxplot(data=eligible,x='mixed_label',y='profile_rmse_median',showfliers=False,color='#D9EAF2',ax=axes[1])
axes[1].set_xlabel('Mixed native labels')
axes[1].set_ylabel('Profile RMSE')
axes[1].set_title('RMSE by label mixing')
for ax in axes: ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
boundary_root=Path('/mnt/newStor/paros/paros_WORK/xintong/results/native_surface_analysis')
boundary_files=sorted(boundary_root.glob('boundary_disagreement_fsaverage6_D*/tables/boundary_agreement_summary.csv'))
if boundary_files:
    boundary=pd.concat([pd.read_csv(p) for p in boundary_files],ignore_index=True)
    boundary=boundary.rename(columns={'region':'source_region'})
    relation=eligible.merge(boundary[['subject_id','hemisphere','source_region','boundary_near_disagreement_pct','internal_disagreement_pct']],on=['subject_id','hemisphere','source_region'],how='left')
    display(relation[['profile_rmse_median','native_anchor_distance_max','boundary_near_disagreement_pct','internal_disagreement_pct']].corr(method='spearman').round(3))
    region_relation=relation.groupby('source_region').agg(profile_rmse=('profile_rmse_median','median'),max_span=('native_anchor_distance_max','median'),boundary_near_disagreement=('boundary_near_disagreement_pct','median'),internal_disagreement=('internal_disagreement_pct','median'),n_groups=('target_vertex_id','size')).sort_values('profile_rmse',ascending=False)
    display(region_relation.round(4))
else:
    print('No fsaverage6 boundary disagreement tables found.')

## Interpretation

Higher multiplicity indicates more native columns collapsed onto one template vertex. Correlation describes profile shape similarity, whereas MAE/RMSE describe absolute MD differences. Native anchor span measures spatial spread on the native white surface. Mixed-label groups indicate that multiple native anatomical labels contributed to the same template vertex. Boundary associations are region-level context unless group-specific boundary distances are computed. Display clipping only limits the plotted range; no groups are removed from the data or tail analyses.